In [1]:
import pandas as pd

prolongations = pd.read_csv('prolongations.csv')
financial_data = pd.read_csv('financial_data.csv')

financial_data['Причина дубля'].drop_duplicates()

0                      NaN
1      первая часть оплаты
2      вторая часть оплаты
19            изменение ЮЛ
77              доп работы
113        основные работы
422           карты, банки
Name: Причина дубля, dtype: object

In [2]:
month_columns = financial_data.iloc[:, 2:-1].columns
financial_data = financial_data[~financial_data[month_columns].isin(['стоп', 'end']).any(axis=1)]
financial_data[month_columns] = financial_data[month_columns].fillna(0)
financial_data[month_columns] = financial_data[month_columns].replace('в ноль', 0)
financial_data[month_columns] = financial_data[month_columns].replace({' ':'', ',':'.', '\xa0':''}, regex=True)
financial_data[month_columns] = financial_data[month_columns].apply(pd.to_numeric)
financial_data.drop('Причина дубля', axis=1, inplace=True)
financial_data = financial_data.groupby(['id', 'Account'], as_index=False).sum()

In [3]:
financial_data = financial_data[~(financial_data[month_columns] == 0).all(axis=1)] 
financial_data[financial_data.duplicated(subset=['id'])].shape[0]

0

In [4]:
financial_data['Account'] = financial_data['Account'].str.strip()
prolongations['AM'] = prolongations['AM'].str.strip()
prolongations['month'] = prolongations['month'].str.strip()
prolongations['month'] = prolongations['month'].str.title()

In [5]:
prolongations.month.unique()

array(['Ноябрь 2022', 'Декабрь 2022', 'Январь 2023', 'Февраль 2023',
       'Март 2023', 'Апрель 2023', 'Май 2023', 'Июнь 2023', 'Июль 2023',
       'Август 2023', 'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023',
       'Декабрь 2023'], dtype=object)

In [6]:
financial_data.columns

Index(['id', 'Account', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2023',
       'Февраль 2023', 'Март 2023', 'Апрель 2023', 'Май 2023', 'Июнь 2023',
       'Июль 2023', 'Август 2023', 'Сентябрь 2023', 'Октябрь 2023',
       'Ноябрь 2023', 'Декабрь 2023', 'Январь 2024', 'Февраль 2024'],
      dtype='object')

In [7]:
months = [
    'Ноябрь 2022', 'Декабрь 2022', 'Январь 2023', 'Февраль 2023',
    'Март 2023', 'Апрель 2023', 'Май 2023', 'Июнь 2023', 'Июль 2023',
    'Август 2023', 'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023',
    'Декабрь 2023', 'Январь 2024', 'Февраль 2024'
]

month_index = {m: i for i, m in enumerate(months)}

df = prolongations.merge(financial_data, left_on=['id', 'AM'], right_on=['id', 'Account'], how='inner')

def get_last_nonzero_shipping(row, month_idx):
    for idx in range(month_idx, -1, -1):
        m = months[idx]
        val = row.get(m, 0)
        if val != 0:
            return val
    return 0

def get_shipping(row, month):
    idx = month_index.get(month, -1)
    if idx == -1:
        return 0
    return get_last_nonzero_shipping(row, idx)

df['month_idx'] = df['month'].map(month_index)

df['shipping_LM'] = df.apply(lambda row: get_shipping(row, row['month']) if pd.notna(row['month_idx']) else 0, axis=1)
df['month_plus_1_idx'] = df['month_idx'] + 1
df['month_plus_2_idx'] = df['month_idx'] + 2
df['shipping_plus_1'] = df.apply(
    lambda row: get_last_nonzero_shipping(row, row['month_plus_1_idx']) if row['month_plus_1_idx'] < len(months) else 0, axis=1)
df['shipping_plus_2'] = df.apply(
    lambda row: get_last_nonzero_shipping(row, row['month_plus_2_idx']) if row['month_plus_2_idx'] < len(months) else 0, axis=1)
results = []


In [8]:
print(df.groupby('month')['shipping_LM'].sum())
print(df.groupby('month')['shipping_plus_1'].sum())

month
Август 2023      2994482.52
Апрель 2023      2867768.00
Декабрь 2022     4725667.65
Декабрь 2023     7695281.85
Июль 2023        1686225.83
Июнь 2023        2299704.85
Май 2023          967580.53
Март 2023        2446580.55
Ноябрь 2022      1531849.00
Ноябрь 2023      2437364.91
Октябрь 2023     1269244.97
Сентябрь 2023    3190500.81
Февраль 2023     1703271.75
Январь 2023      2332055.50
Name: shipping_LM, dtype: float64
month
Август 2023      3014095.30
Апрель 2023      2447710.75
Декабрь 2022     4487331.28
Декабрь 2023     7275858.52
Июль 2023        1432558.51
Июнь 2023        2479624.85
Май 2023         1002620.53
Март 2023        2556726.80
Ноябрь 2022      1533492.00
Ноябрь 2023      2190500.22
Октябрь 2023     1249656.50
Сентябрь 2023    3634272.87
Февраль 2023     1636910.75
Январь 2023      2440525.50
Name: shipping_plus_1, dtype: float64


In [ ]:
for i in range(2, 14):  # с января 2023 (индекс 2) по декабрь 2023 (индекс 13)
    M = months[i]                      # месяц пролонгации
    M_minus_1 = months[i-1]            # месяц завершения проектов для первого коэффициента
    M_minus_2 = months[i-2]            # месяц завершения проектов для второго коэффициента

    # первый коэффициент — проекты, завершившиеся в M-1
    df_coef1 = df[df['month'] == M_minus_1]
    # сумма отгрузки за последний месяц проектов в M-1
    sum_shipping_LM = df_coef1['shipping_LM'].sum()
    # сумма отгрузки проектов из M-1 с отгрузкой в M (первый месяц пролонгации)
    sum_shipping_plus_1 = df_coef1[df_coef1['shipping_plus_1'] > 0]['shipping_plus_1'].sum()
    coef1 = sum_shipping_plus_1 / sum_shipping_LM if sum_shipping_LM > 0 else None

    # второй коэффициент — проекты завершённые в M-2, не пролонгированные в первый месяц (M-1)
    df_coef2 = df[df['month'] == M_minus_2]

    no_renewal_in_M_minus_1 = df_coef2['shipping_plus_1'] == 0

    sum_not_renewed = df_coef2.loc[no_renewal_in_M_minus_1, 'shipping_LM'].sum()
    sum_renewed_second_month = df_coef2.loc[no_renewal_in_M_minus_1 & (df_coef2['shipping_plus_2'] > 0), 'shipping_plus_2'].sum()

    coef2 = sum_renewed_second_month / sum_not_renewed if sum_not_renewed > 0 else None

    # по менеджерам
    for AM in prolongations['AM'].unique():
        df_m1 = df_coef1[df_coef1['AM'] == AM]
        sum_LM_m1 = df_m1['shipping_LM'].sum()
        sum_plus_1_m1 = df_m1[df_m1['shipping_plus_1'] > 0]['shipping_plus_1'].sum()
        coef1_am = sum_plus_1_m1 / sum_LM_m1 if sum_LM_m1 > 0 else None

        df_m2 = df_coef2[df_coef2['AM'] == AM]
        no_renewal_m2 = df_m2['shipping_plus_1'] == 0
        sum_not_renewed_am = df_m2.loc[no_renewal_m2, 'shipping_LM'].sum()
        sum_renewed_second_am = df_m2.loc[no_renewal_m2 & (df_m2['shipping_plus_2'] > 0), 'shipping_plus_2'].sum()
        coef2_am = sum_renewed_second_am / sum_not_renewed_am if sum_not_renewed_am > 0 else None

        results.append({
            'month': M,
            'AM': AM,
            'coef1_prolongation_first_month': coef1_am,
            'coef2_prolongation_second_month': coef2_am
        })

df_results = pd.DataFrame(results)
df_results.head()